# 00　QPSK 验证环境初始化

`03` / `04` / `05` 的每一步都假设"板卡已经就绪"：LMK 出了 256 MHz 与连续 SYSREF、位流已下载、
RFDC 已初始化、DMA 已认对方向。**断电或重启之后这些全都不成立** —— PYNQ 不会自动下位流，
也不会自动配时钟芯片。这本册子把一个可以复现的启动流水线摆出来，跑完它，`03_qpsk_tx_rx` 就能直接出数据。

```
  ┌──────────┐ ①SPI   ┌──────────┐ ②Overlay ┌──────────┐ ③xrfdc  ┌─────────────┐
  │ LMK04828 │───────▶│ 256 MHz  │─────────▶│  v106/   │────────▶│ RFDC 4096   │
  │ 寄存器表 │        │ +SYSREF  │          │ soc.bit  │         │ MSPS + NCO  │
  └──────────┘        └──────────┘          └──────────┘         └──────┬──────┘
                                                                        │④DMA 自检
                                                                 ┌──────▼──────┐
                                                                 │ 环回波峰因数 │
                                                                 └─────────────┘
```

| 步骤 | 函数 | 期望 | 不做 / 做错的典型现象 |
|---|---|---|---|
| 环境 | `preflight()` | root + 时钟脚本齐全 | `/dev/mem`、`spidev` 打不开 |
| ① 时钟 | `ensure_clock()` | PLL2 锁定、SYSREF MUX=3 | DMA 有 IOC，但数据全是 0 |
| ② 位流 | `locate_bit()` / `hw_summary()` | MM 位宽 128、流位宽 32 | TX 出现保持段、RX 峰值偏低约 2% |
| ③ RFDC | `open_hardware()` | DAC state=0xF、ADC 使能 | 星座不转、只有一个点 |
| ④ 自检 | `probe()` | 波峰因数 < 2.5 | ≥3.4 说明 ADC 只回了 I 路 |

`03` / `04` / `05` 的第一格都是调这里的函数，所以四个 notebook 之间**没有隐式依赖**：
重启后先跑本册，或直接从 `03` 的第 0 节开始，结果一样。


## 1　运行环境自检

### 为什么先自检

后面每一步都直接操作 `/dev/mem`（PL 寄存器）、`spidev`（LMK 的 SPI）和 PYNQ 的 `allocate()`（物理内存），
这三样都要求 **root**。Jupyter 默认就是 root，但如果哪天改成别的用户启动，现象会是
`Permission denied` 而不是"没配好"——先确认身份，能省掉一半误判。

同时确认三个时钟脚本还在：`lmk04828_readback.py`（读回）、`lmk04828_set.py`（写表）、
`sysref_start.py`（起 SYSREF）。它们和 TICS 寄存器表都在 `/home/xilinx/`。


In [1]:
import sys, os
QPSK_DIR = "/home/xilinx/qpsk"
if QPSK_DIR not in sys.path:
    sys.path.insert(0, QPSK_DIR)

import qpsk_init as QI

QI.preflight()


kernel : 6.12.40-xilinx-g0900176227cc
python : 3.12.3 → /usr/local/share/pynq-venv/bin/python3
uid    : 0
nb dir : /home/xilinx/qpsk
legacy : /home/xilinx/qpsk
  [PASS] 运行身份是 root                              uid=0
  [PASS] 时钟脚本在位 lmk04828_readback.py            /home/xilinx/lmk04828_readback.py
  [PASS] 时钟脚本在位 lmk04828_set.py                 /home/xilinx/lmk04828_set.py
  [PASS] 时钟脚本在位 sysref_start.py                 /home/xilinx/sysref_start.py
  [PASS] 256 MHz 寄存器表在位                         LMK04828_256m.tics.txt

 1　运行环境
  结果  项目                                         实测 / 说明
  ----------------------------------------------------------------------------
  PASS  运行身份是 root                              uid=0
  PASS  时钟脚本在位 lmk04828_readback.py            /home/xilinx/lmk04828_readback.py
  PASS  时钟脚本在位 lmk04828_set.py                 /home/xilinx/lmk04828_set.py
  PASS  时钟脚本在位 sysref_start.py                 /home/xilinx/sysref_start.py
  PASS  256 MHz 寄存器表在位                        

{'PASS': 5, 'FAIL': 0, 'SKIP': 0}

**应该看到**：`[PASS]` 五项，`uid : 0`。

**不对怎么查**：

| 现象 | 处置 |
|---|---|
| `uid ≠ 0` | Jupyter 要以 root 启动，否则后面所有 MMIO 都会失败 |
| 某个脚本 `FAIL` | 从出厂镜像恢复 `/home/xilinx/<脚本>`，或确认路径 |
| `SPI 节点` 找不到（下一节） | `sudo ls /sys/bus/spi/devices/`，检查设备树的 `compatible` |


## 2　时钟：LMK04828 的 PLL2 与连续 SYSREF

### 为什么这是重启后的第一块绊脚石

LMK 是**外部时钟芯片**，寄存器上电后是空的，必须由软件经 SPI 写一遍。它给出两类信号：

- **256 MHz device clock** → RFDC 内部 PLL ×16 → 4096 MSPS 采样时钟；
- **SYSREF** → 多 tile / 多 converter 的确定性延迟对齐。

时钟没出来时，`xrfdc.init()` **不报错**，`tile_status()` 里 `pll=0`、`state=0`，
DAC 不出数、ADC 不搬数；现象正是"DMA 有 IOC，但缓冲里全是 0"。很容易被误判成 DMA 或 DSP 的 bug。

所以判据放在**时钟层**：`R0x183` bit1 = PLL2 数字锁定（主判据），`R0x139[1:0]=3` = SYSREF 连续输出。
本板 19.2 MHz 晶振直接进 PLL2，PLL1 空载，**`PLL1 LD=0` 是正常的**，不要拿它当故障。


In [2]:
clk = QI.ensure_clock()      # 已锁定时 ~0.2 s 返回；未锁会自动写表 + 起 SYSREF


  读回窗口自检通过（3 字节传输，rx[2] = 数据字节）

=== 配置四线 SPI readback ===
  SPI_LOCK 解锁 (R0x1FFD/R0x1FFE=0x00, R0x1FFF=0x53)
  R0x000 = 0x10 (SPI_3WIRE_DIS=1)
  R0x15F 无条件写回 0x13（写前读到 0x13，脏状态下该读数不可信）
  R0x16E = 0x3B (PLL2_LD_MUX=7 readback, TYPE=3 推挽)
  读回可信度检查通过（R0x171=0xAA / R0x172=0x02）
→ PLL2 已锁定、SYSREF 连续输出，跳过写表
  LMK 现状: 读回可信=True  PLL1 LD=0(0x00)  PLL2 LD=1(0x06)  SYSREF MUX=3
  [PASS] LMK 读回可信                                 R0x171=0xAA / R0x172=0x02
  [PASS] LMK PLL2 锁定（RFDC 时钟源）                 R0x183=0x06
  [PASS] SYSREF 连续输出（MUX=3）                     R0x139 MUX=3
  PLL1 LD=0（本板 PLL1 空载，LD=0 正常，不是故障）

 2　时钟
  结果  项目                                         实测 / 说明
  ----------------------------------------------------------------------------
  PASS  LMK 读回可信                                 R0x171=0xAA / R0x172=0x02
  PASS  LMK PLL2 锁定（RFDC 时钟源）                 R0x183=0x06
  PASS  SYSREF 连续输出（MUX=3）                     R0x139 MUX=3
  --------------------------------------------------------

**应该看到**：`PLL2 LD=1`、`SYSREF MUX=3`、三项 `[PASS]`。

写表那条路（`sane=False`、`PLL2 LD=0` 或 `MUX≠3` 时才走）会打印两个脚本的回显，正常。

**判读表**

| 现象 | 含义 | 处置 |
|---|---|---|
| `sane=False` | 读回通路不可信，寄存器值是假的 | 检查 SPI 速率/节点；必要时 `ensure_clock(force=True)` |
| `PLL2 LD=0` | 没有采样时钟 | 看 `sysref_start.py` 回显；确认 `LMK04828_256m.tics.txt` 存在 |
| `MUX≠3` | SYSREF 不是连续输出 | 同上，脚本会重写 `R0x139` |
| `PLL1 LD=0` | PLL1 空载 | **正常**，忽略 |


## 3　位流：`v106/soc.bit` + 同名 `soc.hwh`

### 为什么两者必须成对、同源

`Overlay("soc.bit")` 做两件事：把位流经 FPGA manager 下进 PL；再解析**同名** `.hwh` 得到 IP 清单、
基地址与参数。只有 `.bit` 时加载"成功"，但 `ip_dict` 里没有地址，后面所有 MMIO 都打到错误的地方；
`.hwh` 与 `.bit` 不同源时，基地址和 `C_FRAME_LEN` 全错，现象是"DMA 有 IOC，数据是垃圾"。

**顺序也必须对**：下位流会复位 PL，所以一律"先 Overlay、后 `xrfdc.init()`"。
本册的 `open_hardware()` 就是按这个顺序做的。

### 为什么要看 `Memory Map Data Width`

TX 侧 DMA 每个时钟要吐出 1 个 32 bit 字（256 MHz × 4 B = 1.024 GB/s）。若 MEMORY MAP 数据位宽也是 32 bit，
每条 MM 读都必须在**一个时钟内**回来，密度 100%，DDR 一有刷新抖动就直接变成 DAC 保持段
（02 §7.6 实测：每 7.789 µs 出现 41 个字的保持，占总时间 2.06%，RX 峰值从 1.0000 MHz 掉到 0.9756 MHz）。
改成 128 bit 后每 4 个流时钟才要 1 拍 MM 读，密度降到 25%，余量全给 FIFO 吸收。


In [3]:
BIT, HWH = QI.locate_bit()          # 默认取 /home/xilinx/jupyter_notebooks/v106/soc.bit
hw = QI.hw_summary(HWH)          # IP 清单 + 两个 DMA 的位宽 + adc2axis 包长


  [PASS] 位流 soc.bit 在位                            /home/xilinx/jupyter_notebooks/v106/soc.bit
  [PASS] 硬件描述 soc.hwh 同源                        /home/xilinx/jupyter_notebooks/v106/soc.hwh
位流 : /home/xilinx/jupyter_notebooks/v106/soc.bit
hwh  : /home/xilinx/jupyter_notebooks/v106/soc.hwh
体积 : 13.2 MB
IP 清单（16）: adc2axis_0, axi_dma_0, axi_dma_1, diver_demo_0, pl_clk_buf_0, rst_rf_clk, smartconnect_0, smartconnect_1, system_ila_0, usp_rf_data_converter_0, xlconcat_0, xlconstant_2, xlslice_2, xpm_cdc_gen_0, xpm_cdc_gen_1, xpm_cdc_gen_2

  IP             方向       MM 数据位宽        流 TDATA 位宽   突发字节       SG
  ----------------------------------------------------------------------------
  axi_dma_0      S2MM/RX  128            32           256        0
  axi_dma_1      MM2S/TX  128            32           64         0

  adc2axis C_FRAME_LEN = 4194303 字（单包上限 16.384 ms）
  [PASS] axi_dma_0 存储器侧位宽 = 128                 C_M_AXI_*_DATA_WIDTH=128
  [PASS] axi_dma_1 存储器侧位宽 = 128                 C_M_AXI_*

**应该看到**（v106 这份 128 位构建）：

```
  IP               方向      MM 数据位宽    流 TDATA 位宽  突发字节   SG
  axi_dma_0        S2MM/RX   128            32             256       0
  axi_dma_1        MM2S/TX   128            32             64        0

  adc2axis C_FRAME_LEN = 4194303 字（单包上限 16.383 ms）
  [PASS] axi_dma_0 存储器侧位宽 = 128
  [PASS] axi_dma_1 存储器侧位宽 = 128
```

**不对怎么查**：

| 现象 | 含义 | 处置 |
|---|---|---|
| `MM 数据位宽 = 32` | 这是旧的 32 位构建 | 换回 128 位位流，或接受 TX 保持段 |
| `没有找到配对的 soc.bit + soc.hwh` | 文件缺失或不同目录 | 放进 `v106/`，或 `export QPSK_BIT=/path/soc.bit` |
| 跑起来 `ip_dict` 为空 | hwh 与 bit 不同名 | 改名成同基名 |


## 4　RFDC 与 NCO

### 为什么 NCO 收发两侧必须同频

DAC 数据路径把复基带乘上 $e^{j2\pi f_c t}$ 再取实部（上变频），ADC 反过来乘 $e^{-j2\pi f_c t}$ 再抽取（下变频）。
两侧用**同一个** $f_c=1000$ MHz 时，基带里的信号原样回到基带；两侧差 $\Delta f$，整个星座会以 $\Delta f$ 匀速旋转，
解调前的频偏估计能吃掉一部分，但残余会直接变成 BER。

`open_hardware()` 做的事：Overlay → `xrfdc.deinit/init` → `set_fs_mhz(4096)` → DAC/ADC NCO = 1000 MHz
→ 按 hwh 参数认 DMA 方向（不靠 `axi_dma_0/1` 的名字）。


In [4]:
lb = QI.open_hardware(BIT)


  overlay 承接 1064 ms，5 个 IP
  DAC0 NCO 已是 1000.000 MHz，沿用
  ADC0 NCO 已是 1000.000 MHz，沿用
  RF 采样率 4096.0 MSPS → 字率 256.000 MHz
  DAC0 enabled=True state=0xF PLL=1
  ADC0 enabled=True state=0xF PLL=1
  [PASS] xRFDC 已就绪                                 is_ready()=True
  [PASS] DAC0 进入 FULL                               state=0xF
  [PASS] ADC0 已使能                                  enabled=True
  [PASS] TX/RX DMA 已定位                             TX@0x80020000 RX@0x80000000

 4　RFDC / DMA
  结果  项目                                         实测 / 说明
  ----------------------------------------------------------------------------
  PASS  xRFDC 已就绪                                 is_ready()=True
  PASS  DAC0 进入 FULL                               state=0xF
  PASS  ADC0 已使能                                  enabled=True
  PASS  TX/RX DMA 已定位                             TX@0x80020000 RX@0x80000000
  ----------------------------------------------------------------------------
  PASS 4 / FAIL 0 / SKIP 0


**应该看到**：`overlay 载入 … ms`、`DAC0/ADC0 NCO 回读 1000.000 MHz`、
`RF 采样率 4096.0 MSPS → 字率 256.000 MHz`、`DAC0 state=0xF`、`ADC0 enabled=True`，四项 `[PASS]`。

> `state` 是 DAC 的 FIFO 满标志，`init()` 刚返回时是 `0x5`，数据通路起转后才变 `0xF`。
> 本函数会自动等最多 2 s（板上实测 0.1~0.2 s 到位），所以不该看到它一直是 `0x5`。

**不对怎么查**：

| 现象 | 处置 |
|---|---|
| `xRFDC 已就绪 FAIL` | 回第 2 节确认 PLL2 锁定 |
| 等 2 s 仍是 `0x5` / `0x0` | 时钟或 SYSREF 未就绪；或 tile 的 PLL 被关掉 |
| 之后星座不转、只有一个点 | 调 `lb.rf_reinit()`（0.1 s，比重下位流快 20 倍） |


## 5　收发自检：波峰因数

### 为什么用波峰因数而不是"看起来对不对"

本板每次下位流后，ADC 有相当概率**只回 I 通道**（虚部恒 0）。此时收到的是实信号 $s+\bar s$：
包络周期性过零，**帧起点、帧间距、相关峰全都正常**，只有星座和比特全错。这个现象极易被误判成 DSP 有 bug。

判据很干净：干净的单边带 QPSK 包络近似恒定，波峰因数（峰值/rms）≈1.6；双边带污染时 ≥3.4。
`probe()` 发 2 帧、抓回、算这个比值；不达标就重跑 `rf_reinit()`，最多 4 次。


In [5]:
cf, x = QI.probe(lb)      # 发 2 帧 QPSK（512 比特），量波峰因数


  自检 0：收 4194299 字，波峰因数 2.10 → 干净
  [PASS] 环回波峰因数 < 2.5                           2.10

 5　收发自检
  结果  项目                                         实测 / 说明
  ----------------------------------------------------------------------------
  PASS  环回波峰因数 < 2.5                           2.10
  ----------------------------------------------------------------------------
  PASS 1 / FAIL 0 / SKIP 0


**应该看到**：第一次就 `波峰因数 1.6x → 干净`，`[PASS] 环回波峰因数 < 2.5`。

若第一次 ≥2.5，会看到它自动 `重跑 RFDC 初始化` 再来一次 —— 这是设计的恢复路径，不是故障。
连续 4 次都 ≥2.5，先查 SMA 环回线和衰减器，再考虑重下位流。


## 6　汇总

把前面几步的判据合成一张表。**全 PASS 再去跑 `03_qpsk_tx_rx.ipynb`。**


In [6]:
def pad(s, w):
    """中文占两列，直接 len() 会让表格歪掉。"""
    return s + " " * max(0, w - sum(2 if ord(ch) > 0x2E80 else 1 for ch in s))


st = lb.status()
rows = [
    ("运行身份 root",             os.geteuid() == 0),
    ("LMK PLL2 锁定",             clk["ld2"] == 1),
    ("SYSREF 连续输出 (MUX=3)",   clk["mux"] == 3),
    ("位流 + hwh 配对",           BIT is not None),
    ("DMA 存储器侧位宽 128",      bool(hw["dmas"]) and all(i["mm"] == "128"
                                for i in hw["dmas"].values())),
    ("RFDC ready",                bool(lb.rf_ready)),
    ("DAC0 FULL (state=0xF)",     st["DAC"].get("state") == 0xF),
    ("ADC0 使能",                 st["ADC"].get("enabled") == 1),
    ("环回波峰因数 < %.1f" % QI.CF_CLEAN, cf < QI.CF_CLEAN),
]

print("%s %s" % (pad("项目", 34), "结果"))
print("-" * 48)
for name, ok in rows:
    print("%s %s" % (pad(name, 34), "PASS" if ok else "FAIL"))
bad = [n for n, ok in rows if not ok]
print("-" * 48)
print("总结：%s" % ("全部通过 —— 下一步打开 03_qpsk_tx_rx.ipynb"
                   if not bad else "未通过：%s（按上面各节的判读表处置）" % "、".join(bad)))


项目                               结果
------------------------------------------------
运行身份 root                      PASS
LMK PLL2 锁定                      PASS
SYSREF 连续输出 (MUX=3)            PASS
位流 + hwh 配对                    PASS
DMA 存储器侧位宽 128               PASS
RFDC ready                         PASS
DAC0 FULL (state=0xF)              PASS
ADC0 使能                          PASS
环回波峰因数 < 2.5                 PASS
------------------------------------------------
总结：全部通过 —— 下一步打开 03_qpsk_tx_rx.ipynb


---

## 下一步

| notebook | 做什么 | 需要本册先跑过吗 |
|---|---|---|
| `03_qpsk_tx_rx` | 生成 9 帧已知数据 → DMA 发 → 立即收 → 存 `qpsk_03_capture.npz` | 不需要（自带第 0 节初始化） |
| `04_qpsk_demod` | 从 npz 里一路解到比特 → 存 `qpsk_04_demod.npz` | 不需要（纯离线） |
| `05_qpsk_verify` | 真值 vs 解调逐比特比对，BER / EVM / 定时 / 频偏五条门限 | 不需要（纯离线） |

命令行等价物（一次跑完 03→04→05 的流程）：

```bash
sudo bash -c '. /etc/profile.d/pynq_venv.sh; python3 /home/xilinx/qpsk/run_test.py --quick'
```

（`PYNQ` 4.0 的 `pl_server` 需要 `XILINX_XRT` 环境变量，`pynq_venv.sh` 会带进来。
Jupyter 内核里天然有这个环境，从 SSH 直接 `sudo` 跑则没有，会报 `No Devices Found`。）

## 中断 / 重启后的最短恢复路径

```python
import qpsk_init as QI
QI.ensure_clock()          # 时钟
BIT, _ = QI.locate_bit()   # 位流
lb = QI.open_hardware(BIT) # RFDC + DMA
```
四行以内，10 秒左右。之后 `lb` 可以直接交给 `lb.loop(words)` 收发。
